## Build Results Fact

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
results_table = f"{catalog_name}.{silver_schema}.results"
sprints_table = f"{catalog_name}.{silver_schema}.sprints"

fact_session_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as f

In [0]:
results_df = spark.read.table(results_table)\
                            .withColumn("session_type", f.lit('RACE'))\
                                .drop("race_name", "race_date", "ingestion_timestamp", "source_file")

sprints_df = spark.read.table(sprints_table)\
                            .withColumn("session_type", f.lit('SPRINT'))\
                                .drop("race_name", "race_date", "ingestion_timestamp", "source_file")

In [0]:
complete_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df =  (
    complete_df.withColumn("is_win", f.col("finish_position") == 1)\
                .withColumn("is_podium", f.col("finish_position").between(1, 3))\
                    .withColumn("has_points", f.col("points") > 0)
)

In [0]:
fact_session_results_df.write.format('delta').mode('overwrite').saveAsTable(fact_session_table)